# 03 — Tokenization Parity & Information Parity: Latin Controls

**Notebook:** `latin/03_tp_ip_sbi_ipi_latin.ipynb`  
**Part of:** `notebooks/latin/` pipeline

This notebook replicates the TP and IP pipeline from the Indic branch, applied to two Latin-script control language pairs: **English→German (en-de)** and **English→Spanish (en-es)**.

Latin-script languages serve as the methodological baseline. Because XLM-RoBERTa was pretrained on large quantities of German and Spanish text, and BLOOM-560M was trained on both, any tokenizer or compression bias observed here is expected to be minimal. Confirming near-unity TP and near-unity IP for these controls validates the measurement instrument: inflated values in the Indic branch reflect genuine script-level bias rather than a systematic artefact of the pipeline.

**Outputs produced:**
- `Information_parity_outputs_all.xlsx` — combined workbook with TP + IP columns for both language pairs
- `latin_tp_ip_summary.csv` — per-language-pair summary for Table 11 (Latin controls row)

**Prerequisites:** `tp_ip_ende_clean.csv` and `tp_ip_enes_clean.csv` in `../data/processed/latin/`.

## Setup

Standard library imports and path configuration. The XLM-RoBERTa tokenizer and BLOOM-560M model are loaded once and reused across both language pairs.

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import (
    XLMRobertaTokenizerFast,
    AutoTokenizer,
    AutoModelForCausalLM,
)

DATA_DIR   = "../data/processed/latin"
OUTPUT_DIR = "../data/processed/latin/tokenization_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 8
MAX_LENGTH = 256

print("Loading XLM-RoBERTa tokenizer ...")
xlmr_tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")
print("Tokenizer loaded.\n")

MODEL_NAME = "bigscience/bloom-560m"
device     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
print(f"Loading : {MODEL_NAME}  (first run downloads ~1.1 GB)")

ip_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ip_tokenizer.padding_side = "right"  # right-pad keeps position 0 as a real token

ip_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
ip_model = ip_model.to(device)
ip_model.eval()

print("Model loaded.\n")

## Input Files

The two CSV files already contain per-sentence scores (BLEU, ChrF, COMET, BERTScore, MQM) alongside the source text, machine translation output, and reference translation. Each file covers multiple MT systems and domains.

In [ ]:
LANG_FILES = {
    "en-de": os.path.join(DATA_DIR, "tp_ip_ende_clean.csv"),
    "en-es": os.path.join(DATA_DIR, "tp_ip_enes_clean.csv"),
}

for lp, path in LANG_FILES.items():
    if not os.path.isfile(path):
        print(f"  ⚠  NOT FOUND: {path}")
        continue
    df = pd.read_csv(path)
    print(f"  {lp}  →  {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"      columns: {list(df.columns)[:8]} ...")

## Tokenization Parity (TP)

TP measures how many more subword tokens the XLM-RoBERTa tokenizer assigns to a target-language sentence relative to its English source.

$$TP = \frac{\text{token count (target)}}{\text{token count (source)}}$$

A value of 1.0 means identical fragmentation; values above 1.0 indicate the target script is split into more pieces than English. For German, compound nouns are sometimes encoded as a single piece, so TP can dip slightly below 1.0. For Spanish, values near 1.0–1.15 are expected.

| Column | Description |
|---|---|
| `target_xlmr_TP` | MT output token count ÷ English source token count |
| `refA_xlmr_TP`   | Reference translation token count ÷ English source token count |

In [ ]:
def tokenize_series(series):
    """Return (token_string, token_count) for every element in a pandas Series."""
    def _tok(text):
        if text is None or (isinstance(text, float) and pd.isna(text)):
            return "", 0
        tokens = xlmr_tokenizer.tokenize(str(text))
        return " | ".join(tokens), len(tokens)
    return series.apply(_tok)


def add_tp_columns(df, source_col="source", target_col="target", ref_col="refA"):
    """Insert token-count and TP columns into df; skip columns already present."""
    src_cnt = f"{source_col}_xlmr_token_count"
    if src_cnt not in df.columns:
        tok = tokenize_series(df[source_col])
        df[f"{source_col}_xlmr_tokens"] = tok.apply(lambda x: x[0])
        df[src_cnt]                     = tok.apply(lambda x: x[1])

    for col, tp_col in [(target_col, "target_xlmr_TP"), (ref_col, "refA_xlmr_TP")]:
        cnt_col = f"{col}_xlmr_token_count"
        if cnt_col not in df.columns:
            tok = tokenize_series(df[col])
            df[f"{col}_xlmr_tokens"] = tok.apply(lambda x: x[0])
            df[cnt_col]              = tok.apply(lambda x: x[1])
        if tp_col not in df.columns:
            src = df[src_cnt].replace(0, float("nan"))
            df[tp_col] = df[cnt_col] / src

    return df


tokenized_dfs = {}
for lp, path in LANG_FILES.items():
    if not os.path.isfile(path):
        continue
    df = pd.read_csv(path)
    df = add_tp_columns(df)
    tokenized_dfs[lp] = df
    print(f"  ✓  {lp}  |  "
          f"Trans. TP mean = {df['target_xlmr_TP'].mean():.3f}  "
          f"| Ref. TP mean = {df['refA_xlmr_TP'].mean():.3f}")

print("\nTP interpretation:")
print("  1.0 = same token count as English  (no tokenizer bias)")
print("  0.9 = slightly fewer tokens  (German compound compression)")
print("  1.2 = 20 % more tokens than English  (moderate fragmentation)")

## Information Parity (IP)

IP captures whether the language model compresses target-language text as efficiently as English, using BLOOM-560M's negative log-likelihood (NLL) as the compressibility proxy.

$$IP = \frac{NLL_{\text{English source}}}{NLL_{\text{target}}}$$

A value of 1.0 means the model finds the target equally predictable to English; values below 1.0 mean the target is harder to predict. For German and Spanish — both present in BLOOM's pretraining corpus — values of roughly 0.6–1.0 are expected, substantially higher than the Indic languages (which cluster around 0.3–0.5).

**Total NLL is used, not mean per-token NLL.** Using mean NLL would conflate IP with TP, since more tokens per sentence would artificially lower the per-token average. Total NLL keeps the two metrics conceptually independent.

In [ ]:
def compute_nll(texts):
    """Return per-sentence total cross-entropy NLL for each string in `texts`.

    Uses total NLL (sum over real tokens) so that IP and TP remain independent.
    Right-padding ensures position 0 always holds a real token, preventing the
    position-embedding corruption that occurs with left-padding in batches.
    """
    results = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch_raw = texts[i : i + BATCH_SIZE]
        valid_idx, valid_texts = [], []
        for j, t in enumerate(batch_raw):
            if t and not (isinstance(t, float) and pd.isna(t)):
                valid_idx.append(j)
                valid_texts.append(str(t))

        batch_res = [float("nan")] * len(batch_raw)
        if not valid_texts:
            results.extend(batch_res)
            continue

        enc = ip_tokenizer(
            valid_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        with torch.no_grad():
            logits = ip_model(**enc).logits

        for k, oi in enumerate(valid_idx):
            ids   = enc["input_ids"][k]
            mask  = enc["attention_mask"][k]
            sl    = logits[k, :-1, :]
            sl_id = ids[1:]
            sl_m  = mask[1:].bool()
            if sl_m.sum() == 0:
                batch_res[oi] = float("nan")
                continue
            total_nll = F.cross_entropy(sl[sl_m], sl_id[sl_m], reduction="sum")
            batch_res[oi] = total_nll.item()

        results.extend(batch_res)
    return results

## Computing IP Across Both Language Pairs

For each CSV the English source NLL is computed once (the shared denominator). IP is then inserted as a per-row ratio. The cell is safe to re-run — it checks whether IP columns already exist before inserting them.

In [ ]:
IP_PAIRS = [
    ("target_xlmr_IP", "target"),
    ("refA_xlmr_IP",   "refA"),
]

for lp, df in tokenized_dfs.items():
    print(f"Processing {lp} ...")
    src_nll = (
        pd.Series(compute_nll(df["source"].fillna("").tolist()))
        .replace(0, float("nan"))
    )
    if "source_NLL" not in df.columns:
        df.insert(df.columns.get_loc("source") + 1, "source_NLL", src_nll.values)

    for ip_col, text_col in IP_PAIRS:
        if text_col not in df.columns:
            print(f"  '{text_col}' not found — skipped")
            continue
        if ip_col in df.columns:
            continue
        nll_tgt = (
            pd.Series(compute_nll(df[text_col].fillna("").tolist()))
            .replace(0, float("nan"))
        )
        df.insert(df.columns.get_loc(text_col) + 1, ip_col, (src_nll / nll_tgt).values)

    tokenized_dfs[lp] = df
    print(f"  ✓  {lp}  |  "
          f"Trans. IP mean = {df['target_xlmr_IP'].mean():.3f}  "
          f"| Ref. IP mean = {df['refA_xlmr_IP'].mean():.3f}\n")

print("IP interpretation:")
print("  1.0 = model compresses target as well as English")
print("  0.7 = model finds target 30 % harder than English")
print("  0.5 = model finds target much harder than English")

## Summary Statistics — Table 11 (Latin Controls Row)

Aggregate TP and IP across all systems and domains to produce the per-language-pair summary used in Table 11.

In [ ]:
rows = []
for lp, df in tokenized_dfs.items():
    rows.append({
        "Language pair":    lp,
        "Script":           "Latin",
        "Trans. TP (mean)": round(df["target_xlmr_TP"].mean(), 3),
        "Trans. IP (mean)": round(df["target_xlmr_IP"].mean(), 3),
        "Ref. TP (mean)":   round(df["refA_xlmr_TP"].mean(),   3),
        "Ref. IP (mean)":   round(df["refA_xlmr_IP"].mean(),   3),
        "N":                len(df),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

summary_path = os.path.join(OUTPUT_DIR, "latin_tp_ip_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved → {summary_path}")

## Saving Tokenized CSVs

Each augmented DataFrame is written back to the output directory with TP and IP columns appended.

In [ ]:
for lp, df in tokenized_dfs.items():
    out_name = lp.replace("-", "") + "_tp_ip.csv"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df.to_csv(out_path, index=False)
    print(f"  ✓  Saved: {out_path}  ({len(df)} rows)")

## Exporting to Excel

All augmented CSVs are combined into a single Excel workbook (`Information_parity_outputs_all.xlsx`) with one sheet per language pair. This file is structurally identical to the workbook produced by `05_information_parity.ipynb` in the Indic branch, enabling direct cross-branch comparison in downstream analysis.

In [ ]:
excel_path  = os.path.join(OUTPUT_DIR, "Information_parity_outputs_all.xlsx")
csv_outputs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for csv_path in csv_outputs:
        sheet = os.path.splitext(os.path.basename(csv_path))[0][:31]
        pd.read_csv(csv_path).to_excel(writer, sheet_name=sheet, index=False)

print(f"✓  Combined Excel saved → {excel_path}")

## References

- **XLM-RoBERTa (tokenizer backbone):** Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). Unsupervised Cross-lingual Representation Learning at Scale. *ACL 2020*. <https://arxiv.org/abs/1911.02116>

- **BLOOM-560M (NLL model):** BigScience Workshop. (2022). BLOOM: A 176B-Parameter Open-Access Multilingual Language Model. <https://arxiv.org/abs/2211.05100>

- **Tokenization bias in multilingual models:** Kanjirangat, V., Samardžić, T., Dolamic, L., & Rinaldi, F. (2025). Tokenization and Representation Biases in Multilingual Models on Dialectal NLP Tasks. *EMNLP 2025*, pp. 23992–24010. <https://arxiv.org/abs/2509.20045>

- **Cross-lingual tokenization fairness:** Foroutan, N., Meister, C., Paul, D., Niklaus, J., Ahmadi, S., Bosselut, A., & Sennrich, R. (2025). Parity-Aware Byte-Pair Encoding: Improving Cross-lingual Fairness in Tokenization. *arXiv:2508.04796*. <https://arxiv.org/abs/2508.04796>

- **COMET (wmt22-comet-da):** Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*. <https://aclanthology.org/2020.emnlp-main.213>

- **IndicMT Eval dataset:** Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., & Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*. <https://aclanthology.org/2023.acl-long.795>